# Vocabulary Evaluator (Early Release)

**The Vocabulary Evaluator** gives developers the fine-grained insight they need but can’t get from traditional tools. It helps determine whether texts use words that align with grade-level expectations and support growth in academic language. This ensures students are consistently exposed to the kinds of vocabulary that build knowledge and enable them to fully engage with grade-level texts.

By understanding what makes a text difficult for a student to read, edtech companies and educators are better equipped to ensure students get the right text for their needs, along with the right instructional supports.

You can use this evaluator to help ensure AI-generated texts are sufficiently complex for the grade level and their intended purpose.

1. It estimates a student’s background knowledge given the selected grade level.
2. It uses the background knowledge estimate as a starting point to evaluate the complexity of a passage’s vocabulary.

### Install & Load necessary packages

In [ ]:
%pip install -qU pydantic textstat langchain langchain_openai langchain-google-genai

In [ ]:
# Load packages
import getpass
import os

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from textstat import textstat as ts

### Set up the evaluator's model and prompts

In [ ]:
from prompts import vocab_prompts as prompts

# Set your api keys in your environment, .env file, or enter when prompted.
# os.environ['GOOGLE_API_KEY'] = 'YOUR API KEY'
# os.environ['OPENAI_API_KEY'] = 'YOUR API KEY'
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

# Define the model to be used for vocabulary complexity
VOCAB_MODEL = "gemini-2.5-pro"
VOCAB_TEMPERATURE = 0
vocab_complexity_model = ChatGoogleGenerativeAI(
    model=VOCAB_MODEL, temperature=VOCAB_TEMPERATURE
)

# Define the model to be used for student background knowledge generation
BK_MODEL = "gpt-4o-2024-11-20"
BK_TEMPERATURE = 0
student_bk_model = ChatOpenAI(model=BK_MODEL, temperature=BK_TEMPERATURE)

### Set up student background knowledge generator

In [ ]:
def get_background_knowledge_assumption(text, grade):
    """Use the background knowledge prompt from the prompts file."""
    prompt = prompts.bk_prompt.format(text=text, grade=grade)

    return student_bk_model.invoke(prompt).content

### Set up the input variables and output format

In [ ]:
class Output(BaseModel):
    tier_2_words: str = Field(description="List of Tier 2 words")
    tier_3_words: str = Field(description="List of Tier 3 words")
    archaic_words: str = Field(description="List of Archaic words")
    other_complex_words: str = Field(description="List of Other Complex words")
    complexity_score: str = Field(
        description="the complexity of the text, one of: slightly complex, moderately complex, very complex, or exceedingly complex"
    )
    reasoning: str = Field(description="your reasoning for your answer")


prompt_vars = {
    "inputVars": [
        "text",
        "student_grade_level",
        "student_background_knowledge",
        "fk_level",
    ],
    "outputParser": JsonOutputParser(pydantic_object=Output),
}

### Helper functions

In [ ]:
import textwrap


def calculate_fk_score(text) -> float:
    """
    Calculate the Flesch-Kincaid Grade Level
    """
    fk_score = round(ts.flesch_kincaid_grade(text), 2)

    return fk_score


def prepare_text_for_complexity_prediction(text, grade):
    """
    Enrich the text and grade given by user with additional features for complexity prediction.
    """
    dataset = {
        "text": text,
        "student_grade_level": grade,
        "fk_level": calculate_fk_score(text),
        "student_background_knowledge": get_background_knowledge_assumption(
            text, grade
        ),
    }

    return dataset


def prettify_vocab_complexity_output(vocab_complexity_output):
    output = f"""
        ========================= Complexity Score ========================
        {vocab_complexity_output['complexity_score']}

        ========================= Complexity Score Reasoning ==============
        {textwrap.fill(vocab_complexity_output['reasoning'], width=80)}

        ========================  Complex words  ==========================
        * Tier 2 words: {textwrap.fill(vocab_complexity_output['tier_2_words'], width=65)}
        * Tier 3 words: {textwrap.fill(vocab_complexity_output['tier_3_words'], width=65)}
        * Archaic words: {textwrap.fill(vocab_complexity_output['archaic_words'], width=65)}
        * Other complex words: {textwrap.fill(vocab_complexity_output['other_complex_words'], width=60)}"""

    print(textwrap.dedent(output).strip())

### Define the main evaluation function

In [ ]:
def predict_text_complexity_level(text, grade):
    """
    Predict the text complexity level as well as the complex words and reasoning.
    """

    dataset = prepare_text_for_complexity_prediction(text, grade)

    # Prompts imported from prompts file
    messages = [
        SystemMessage(content=prompts.SYSTEM_PROMPT),
        HumanMessagePromptTemplate.from_template(prompts.USER_PROMPT),
    ]

    # Prepare chat prompt
    prompt = ChatPromptTemplate(
        messages,
        input_variables=prompt_vars["inputVars"],
        partial_variables={
            "format_instructions": prompt_vars["outputParser"].get_format_instructions()
        },
    )
    # Invoke the chain
    chain = prompt | vocab_complexity_model | JsonOutputParser()

    # return output
    output = chain.invoke(dataset)

    return output

# Test out examples

In [ ]:
# Add your text & the grade level you want to evaluate for vocabulary complexity

# Clear ID = 2204
text = """
Polo went on a 24-year trip to China with his father and uncle during the Mongol Dynasty. He left Venice at the age of 17 on a boat that went through the Mediterranean Sea, Ayas, Tabriz and Kerman. Then he travelled across Asia getting as far as Beijing. On the way there he had to go over mountains and through terrible deserts, across hot burning lands and places where the cold was horrible. He served in Kublai Khan's court for 17 years. He left the Far East and returned to Venice by sea. There was sickness on board and 600 passengers and crew died and some say pirates attacked. Nevertheless, Marco Polo survived it all.
Some scholars believe that while Marco Polo did go to China, he did not go to all of the other places described in his book. He brought noodles back from China and the Italians came up with different sizes and shapes and called it pasta. Polo returned to Venice with treasures like ivory, jade, jewels, porcelain and silk.
His father had borrowed money and bought a ship. He became wealthy because of his trading in the near East.
"""

grade_level = 3

vocabulary_complexity_output = predict_text_complexity_level(text, grade_level)

# Pretty Print the output
prettify_vocab_complexity_output(vocabulary_complexity_output)

You can copy or edit the above cell to test out different texts and grade levels.